In [0]:
# Databricks Notebook: 01_Bronze_Ingestion


# Read all CSV files from raw container
raw_path = "abfss://raw@sarveshbankinglake00189.dfs.core.windows.net/"


accounts_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}accounts.csv")
atm_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}atm_transactions.csv")
branches_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}branches.csv")
credit_cards_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}credit_cards.csv")
customers_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}customers.csv")
employees_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}employees.csv")
fraud_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}fraud_alerts.csv")
kyc_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}kyc_documents.csv")
loans_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}loans.csv")
transactions_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}transactions.csv")


print("✅All CSV files loaded successfully")


In [0]:
%python
# Write each DataFrame as Delta table in Bronze layer
bronze_path = "abfss://bronze@sarveshbankinglake00189.dfs.core.windows.net/"


# Accounts
accounts_df.write.format("delta").mode("overwrite").save(f"{bronze_path}accounts")


# ATM Transactions
atm_df.write.format("delta").mode("overwrite").save(f"{bronze_path}atm_transactions")


# Branches
branches_df.write.format("delta").mode("overwrite").save(f"{bronze_path}branches")


# Credit Cards
credit_cards_df.write.format("delta").mode("overwrite").save(f"{bronze_path}credit_cards")


# Customers
customers_df.write.format("delta").mode("overwrite").save(f"{bronze_path}customers")


# Employees
employees_df.write.format("delta").mode("overwrite").save(f"{bronze_path}employees")


# Fraud Alerts
fraud_df.write.format("delta").mode("overwrite").save(f"{bronze_path}fraud_alerts")


# KYC Documents
kyc_df.write.format("delta").mode("overwrite").save(f"{bronze_path}kyc_documents")


# Loans
loans_df.write.format("delta").mode("overwrite").save(f"{bronze_path}loans")


# Transactions
transactions_df.write.format("delta").mode("overwrite").save(f"{bronze_path}transactions")


print("✅ All tables written to Bronze layer!")


In [0]:
%sql
-- Create external tables pointing to Bronze Delta files
CREATE TABLE banking_catalog.bronze.accounts
USING DELTA
LOCATION 'abfss://bronze@sarveshbankinglake00189.dfs.core.windows.net/accounts/';




In [0]:
%python
# Databricks Notebook: 02_Silver_Transformation


from pyspark.sql.functions import *
from pyspark.sql.types import *


# Read from Bronze
accounts_bronze = spark.table("banking_catalog.bronze.accounts")


# Data Cleaning Operations
accounts_silver = accounts_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_source", lit("accounts.csv"))


# Remove duplicates
accounts_silver = accounts_silver.dropDuplicates(["account_id"])


# Write to Silver
accounts_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/accounts")


print(f"✅ Accounts Silver: {accounts_silver.count()} records")



In [0]:

%python
# Read from Bronze
customers_bronze = spark.table("banking_catalog.bronze.customers")


# Data Cleaning
customers_silver = customers_bronze \
    .withColumn("name", trim(col("name"))) \
    .withColumn("gender", when(col("gender").isin(['M', 'F']), col("gender")).otherwise('U')) \
    .withColumn("phone", regexp_replace(col("phone"), "[^0-9]", "")) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("state", upper(trim(col("state")))) \
    .withColumn("kyc_status", when(col("kyc_status").isin(['Y', 'N']), col("kyc_status")).otherwise('N')) \
    .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd")) \
    .withColumn("created_date", to_date(col("created_date"), "yyyy-MM-dd")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_source", lit("customers.csv"))


# Handle NULLs
customers_silver = customers_silver.fillna({
    "phone": "0000000000",
    "email": "unknown@email.com",
    "kyc_status": "N"
})


# Remove duplicates
customers_silver = customers_silver.dropDuplicates(["customer_id"])


# Write to Silver
customers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/customers")


print(f"✅ Customers Silver: {customers_silver.count()} records")

In [0]:
# Problem 8: Clean and Transform Transactions Data

# Read from Bronze
transactions_bronze = spark.table("banking_catalog.bronze.transactions")

# Data Cleaning
transactions_silver = transactions_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Filter invalid transactions (negative amount or amount > 100000)
transactions_silver = transactions_silver \
    .filter(col("amount") >= 0) \
    .filter(col("amount") <= 100000) \
    .dropDuplicates(["transaction_id"])

# Write to Silver
transactions_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/transactions")

print(f"✅ Transactions Silver: {transactions_silver.count()} records")

In [0]:
# Problem 9: Clean and Transform Loans Data

# Read from Bronze
loans_bronze = spark.table("banking_catalog.bronze.loans")

# Data Cleaning
loans_silver = loans_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'SAVINGS', 'CURRENT']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Remove duplicates
loans_silver = loans_silver.dropDuplicates(["loan_id"])

# Write to Silver
loans_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/loans")

print(f"✅ Loans Silver: {loans_silver.count()} records")

In [0]:
# Problem 10: Clean and Transform Fraud Alerts Data

# Read from Bronze
fraud_bronze = spark.table("banking_catalog.bronze.fraud_alerts")

# Data Cleaning
fraud_silver = fraud_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("is_high_risk", when(col("amount") > 40000, lit("Y")).otherwise(lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Remove duplicates
fraud_silver = fraud_silver.dropDuplicates(["alert_id"])

# Write to Silver
fraud_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/fraud_alerts")

print(f"✅ Fraud Alerts Silver: {fraud_silver.count()} records")

In [0]:
%sql
-- Problem 11: Create Silver Tables in Unity Catalog
--Create Silver tables

CREATE TABLE banking_catalog.silver.accounts
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/accounts/';

CREATE TABLE banking_catalog.silver.customers
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/customers/';

CREATE TABLE banking_catalog.silver.transactions
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/transactions/';

CREATE TABLE banking_catalog.silver.loans
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/loans/';

CREATE TABLE banking_catalog.silver.fraud_alerts
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/fraud_alerts/';

CREATE TABLE banking_catalog.silver.atm_transactions
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/atm_transactions/';

CREATE TABLE banking_catalog.silver.branches
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/branches/';

CREATE TABLE banking_catalog.silver.credit_cards
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/credit_cards/';

CREATE TABLE banking_catalog.silver.kyc_documents
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/kyc_documents/';

CREATE TABLE banking_catalog.silver.employees
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/employees/';

In [0]:
#PART 3: GOLD LAYER - Business Insights & Analytics
#Problem 12: Create Dim_Customers Table (SCD Type 2)%

# Databricks Notebook: 03_Gold_Star_Schema

from pyspark.sql.window import Window

# Read Silver tables
customers_silver = spark.table("banking_catalog.silver.customers")
accounts_silver = spark.table("banking_catalog.silver.accounts")
transactions_silver = spark.table("banking_catalog.silver.transactions")
loans_silver = spark.table("banking_catalog.silver.loans")
fraud_silver = spark.table("banking_catalog.silver.fraud_alerts")
atm_silver = spark.table("banking_catalog.silver.atm_transactions")
branches_silver = spark.table("banking_catalog.silver.branches")

In [0]:
from pyspark.sql.functions import *
# Create Dim_Customers with SCD Type 2
dim_customers = customers_silver \
    .withColumn("customer_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit("9999-12-31").cast("date")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("full_name", concat(col("name"), lit(" (ID: "), col("customer_id"), lit(")"))) \
    .withColumn("age", datediff(current_date(), col("dob")) / 365.25) \
    .withColumn("age_group", 
        when(col("age") < 30, "Young")
        .when(col("age") < 50, "Middle")
        .otherwise("Senior"))

# Write to Gold
dim_customers.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_customers")

print(f"✅ Dim_Customers: {dim_customers.count()} records")

In [0]:
#Problem 13: Create Dim_Accounts Table

# Create Dim_Accounts
dim_accounts = accounts_silver \
    .withColumn("account_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("is_current", lit(True)) \
    .withColumn("account_age_days", datediff(current_date(), col("date"))) \
    .withColumn("balance_category",
        when(col("amount") < 10000, "Low")
        .when(col("amount") < 50000, "Medium")
        .otherwise("High"))

# Write to Gold
dim_accounts.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_accounts")

print(f"✅ Dim_Accounts: {dim_accounts.count()} records")

In [0]:
#Problem 14: Create Dim_Date Table
# Create Date Dimension (2020-2025)
def create_date_dimension(start_date, end_date):
    dates = spark.sql(f"""
        SELECT EXPLODE(SEQUENCE(
            TO_DATE('{start_date}'), 
            TO_DATE('{end_date}'), 
            INTERVAL 1 DAY
        )) AS date
    """)
    
    date_dim = dates.select(
        col("date"),
        year("date").alias("year"),
        quarter("date").alias("quarter"),
        month("date").alias("month"),
        dayofmonth("date").alias("day"),
        dayofweek("date").alias("day_of_week"),
        date_format("date", "EEEE").alias("day_name"),
        date_format("date", "MMMM").alias("month_name"),
        when(dayofweek("date").isin([1, 7]), "Weekend").otherwise("Weekday").alias("weekday_indicator")
    ).withColumn("date_sk", monotonically_increasing_id())
    
    return date_dim

date_dim_df = create_date_dimension("2020-01-01", "2025-12-31")

date_dim_df.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_date")

print(f"✅ Dim_Date: {date_dim_df.count()} records")

In [0]:
#Problem 15: Create Fact_Transactions

# Create Fact_Transactions with surrogate keys
fact_transactions = transactions_silver \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), transactions_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "transaction_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("transaction_type"),
        "branch",
        "remarks",
        "flag",
        "year",
        "month",
        "quarter"
    )

# Write to Gold
fact_transactions.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_transactions")

print(f"✅ Fact_Transactions: {fact_transactions.count()} records")

In [0]:
#Problem 16: Create Fact_Loans

# Create Fact_Loans
fact_loans = loans_silver \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), loans_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "loan_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("loan_type"),
        "branch",
        "remarks",
        "flag"
    )

# Write to Gold
fact_loans.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_loans")

print(f"✅ Fact_Loans: {fact_loans.count()} records")

In [0]:
#Problem 17: Create Fact_Fraud_Alerts

# Create Fact_Fraud_Alerts
fact_fraud = fraud_silver \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), fraud_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "alert_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("alert_type"),
        "branch",
        "remarks",
        "flag",
        "is_high_risk"
    )

# Write to Gold
fact_fraud.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_fraud_alerts")

print(f"✅ Fact_Fraud_Alerts: {fact_fraud.count()} records")

In [0]:
%sql

-- Create Gold Tables in Unity Catalog
-- Create Gold tables
CREATE TABLE banking_catalog.gold.dim_customers
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_customers/';

CREATE TABLE banking_catalog.gold.dim_accounts
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_accounts/';

CREATE TABLE banking_catalog.gold.dim_date
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_date/';

CREATE TABLE banking_catalog.gold.fact_transactions
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_transactions/';

CREATE TABLE banking_catalog.gold.fact_loans
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_loans/';

CREATE TABLE banking_catalog.gold.fact_fraud_alerts
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_fraud_alerts/';